In [ ]:
import pygame
import random
import time
import os

pygame.init()

screen = pygame.display.set_mode((800, 600))
pygame.display.set_caption("Catch game")

player_img = pygame.image.load("basket.png")
player_img = pygame.transform.scale(player_img, (100, 50))
apple_img_1 = pygame.image.load("1apple.png")
apple_img_1 = pygame.transform.scale(apple_img_1, (50, 50))
apple_img_5 = pygame.image.load("5apple.png")
apple_img_5 = pygame.transform.scale(apple_img_5, (50, 50))
enemy_img = pygame.image.load("explosion.png")
enemy_img = pygame.transform.scale(enemy_img, (50, 50))
mult_img = pygame.image.load("green_apple.jpg")
mult_img = pygame.transform.scale(mult_img, (50, 50))

player_x = 350
player_y = 500
score = 0
start_time = time.time()
duration = 60
high_score_file = "highscores.txt"
mult_active = False
mult_end_time = 0
multiplier = 2 

def get_time_remaining():
    elapsed_time = time.time() - start_time
    return max(0, int(duration - elapsed_time))

clock = pygame.time.Clock()

choices_list = []
choices_list.append(enemy_img)
choices_list.append(mult_img)
for _ in range(3):
    choices_list.append(apple_img_1)
    choices_list.append(apple_img_5)

apples = [
    {"img" : random.choice(choices_list),
     "x" : random.randint(0, 750), 
     "y" : random.randint(-600, -50),
     "speed" : random.randint(3, 7)}
     for _ in range(5)
]

# high score helpers
def load_high_score():
    if os.path.exists(high_score_file):
        with open(high_score_file, "r") as file: # r for read
            return int(file.read().strip() or 0)
    return 0

def save_high_score(score):
    with open(high_score_file, "w") as file: # w for write
        file.write(str(score))

high_score = load_high_score()
font = pygame.font.Font(None, 36)

running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    keys = pygame.key.get_pressed()
    if keys[pygame.K_LEFT] and player_x > 0:
        player_x -= 7
    if keys[pygame.K_RIGHT] and player_x < 800 - 100:
        player_x += 7

    time_remaining = get_time_remaining()
    if mult_active and time.time() >= mult_end_time:
        mult_active = False

    if time_remaining == 0:
        print(f"Your final score is {score}")
        if score > high_score:
            high_score = score
            save_high_score(high_score)
            print("New high score!")
        running = False

    screen.fill((255, 255, 255))

    player_rect = pygame.Rect(player_x, player_y, 100, 50)
    # handle apples
    for apple in apples:
        apple["y"] += apple["speed"]
        apple_rect = pygame.Rect(apple["x"], apple["y"], 50, 50)
        collision = player_rect.colliderect(apple_rect)
        if apple["y"] > 600 or collision: # apple has gone off screen OR collided
            if collision:
                if apple["img"] == apple_img_1:
                    if mult_active:
                        score += 1 * multiplier
                    else:
                        score += 1
                elif apple["img"] == apple_img_5:
                    if mult_active:
                        score += 5 * multiplier
                    else:
                        score += 5
                elif apple["img"] == enemy_img:
                    score -= random.randint(1, 50)
                elif apple["img"] == mult_img:
                    mult_active = True
                    mult_end_time = time.time() + 5
            apple["y"] = random.randint(-600, -50)
            apple["x"] = random.randint(0, 750)
            # bias towards lower scoring apples
            apple["img"] = random.choice(choices_list)
            
        screen.blit(apple["img"], (apple["x"], apple["y"]))

    screen.blit(player_img, (player_x, player_y))

    score_text = font.render(f"Score: {score}", True, (0, 0, 0))
    screen.blit(score_text, (10, 10))

    timer_text = font.render(f"Score: {time_remaining}s", True, (0, 0, 0))
    screen.blit(timer_text, (10, 50))

    high_score_text = font.render(f"High score: {high_score}", True, (0, 0, 0))
    screen.blit(high_score_text, (10, 90))

    if mult_active:
        mult_text = font.render(f"Multiplier active: x{multiplier}", True, (0, 0, 0))
        screen.blit(mult_text, (10, 130))
    
    pygame.display.flip()

    clock.tick(60)

pygame.quit()
# TODO: 
# higher ratio of 5-point apples at a certain score?
# difficulty levels
# main menu screen
# multiplayer
# multiplier gives speed buff and enemy immunity

Your final score is 78


In [ ]:
# pastebin.com/50f99qGP